In [1]:
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
import torch
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from data_utils import HarmonicBiLSTMDataset, harmonic_bilstm_collate_fn

from models_BiLSTM import HarmonyBiLSTM
from generate_utils import load_AttnFiLMSEModel

import pickle

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the dataset and the tokenizer
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
print(gjt_train[0].keys())
print(gjt_train[0]['graph_ready_object'].print_info())

dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity', 'graph_ready_object'])
Number of bars: 16
No segment graph created yet.
Bar 1:
Bar token positions: [1, 2, 3, 4]
Number of chord objects in bar: 2
Chord object 1:
Chord label: A:min7
Pitch classes: [0, 4, 7, 9, 11]
Root: 9
Chord ID: 276
Bar Positions: [0, 1]
Token Positions: [1, 2]
Melody PCs: [array([], dtype=int64), array([ 9, 11])]
Graph Features:
tensor([[0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 1., 0., 1., 1.]])
BiLSTM Features
tensor([1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 1.])
Chord object 2:
Chord label: F#:hdim7
Pitch classes: [0, 4, 6, 9]
Root: 6
Chord ID: 194
Bar Positions: [2, 3]
Token Positions: [3, 4]
Melody PCs: [array([0]), array([0, 4])]
Graph Features:
tensor([[0., 0., 1

In [4]:
gjt_train[0]['graph_ready_object'].bar_objects[0].print_info()

Bar token positions: [1, 2, 3, 4]
Number of chord objects in bar: 2
Chord object 1:
Chord label: A:min7
Pitch classes: [0, 4, 7, 9, 11]
Root: 9
Chord ID: 276
Bar Positions: [0, 1]
Token Positions: [1, 2]
Melody PCs: [array([], dtype=int64), array([ 9, 11])]
Graph Features:
tensor([[0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 1., 0., 1., 1.]])
BiLSTM Features
tensor([1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 1.])
Chord object 2:
Chord label: F#:hdim7
Pitch classes: [0, 4, 6, 9]
Root: 6
Chord ID: 194
Bar Positions: [2, 3]
Token Positions: [3, 4]
Melody PCs: [array([0]), array([0, 4])]
Graph Features:
tensor([[0., 0., 1., 0., 0., 1., 1., 0.],
        [0., 0., 0., 1., 0., 1., 1., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 1., 0., 0.]])
BiLSTM Features
tensor([1.,

In [5]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

bilstm_model = HarmonyBiLSTM()
bilstm_model.to(device)

# load the model
model = load_AttnFiLMSEModel(
    tokenizer=tokenizer,
    guidance_dim=bilstm_model.output_dim,
    device=device
)
model.freeze_base()

In [6]:
dataset = HarmonicBiLSTMDataset(gjt_train, tokenizer, model)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=harmonic_bilstm_collate_fn)

In [7]:
batch = next(iter(dataloader))
print(batch.keys())

dict_keys(['pianoroll', 'real_harmony_ids', 'recomposed_harmony_ids', 'random_harmony_ids', 'real_bilstm', 'real_lengths', 'recomposed_bilstm', 'recomposed_lengths', 'random_bilstm', 'random_lenghts', 'mask_token_positions'])


In [8]:
y = bilstm_model(batch['real_bilstm'], batch['real_lengths'])

In [9]:
print(y.shape)

torch.Size([16, 128])


In [10]:
print(batch['pianoroll'].shape)
constraints = batch['real_harmony_ids'].clone()
constraints[batch['mask_token_positions']] = tokenizer.mask_token_id
print(constraints.shape)
print(constraints.dtype)

torch.Size([16, 80, 13])
torch.Size([16, 80])
torch.int64


In [11]:
real_constraints = batch['real_harmony_ids'].clone()
real_constraints[batch['mask_token_positions']] = tokenizer.mask_token_id

In [12]:
real_logits = model(
    batch['pianoroll'].to(model.device),
    real_constraints.to(model.device),
    y.to(model.device)
)

In [13]:
print(real_logits.shape)

torch.Size([16, 80, 355])
